# B2.14 · Bonus — Google Mantis, the pipeline in production

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.13 · Attesting control intent for agents and MCP servers](https://spbreed.github.io/cyber-commons/lessons/B2.13.html)**.

| | |
|---|---|
| Tools used | Google Mantis, OpenGrep, GLM-4.6, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Map Mantis onto the 15 stages, parse its two output shapes, and score a sample against a held-out key.

**Why a security engineer needs it.** A reference implementation is adopted as a product, and its outputs are trusted without an eval. The control it builds is: map Mantis's stages onto the pipeline you built, then score it with your own held-out key before trusting it.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Somebody has already built this pipeline and published what happened. Reading it is worth an afternoon; adopting it without scoring it against a held-out key is how a reference implementation becomes a dependency you cannot evaluate.

> **At CyberTravels.** Somebody else has already built this pipeline and published what happened. Adopting it without scoring it against a held-out key is how a reference implementation becomes a dependency CyberTravels cannot evaluate.

## 2 · The framework

```
   published pipeline            your pipeline
   +------------------+          +------------------+
   | stages 1..15     |  map ->  | stages 1..15     |
   +------------------+          +------------------+
            |
       score it against a HELD-OUT key
            |
   +--------v---------+
   | adopt / adapt /  |
   | leave it alone   |
   +------------------+

   a reference implementation is a starting point you evaluate
```

**Bonus.** You have now built all fifteen stages. This lesson looks at a real
implementation of the same pipeline — **[Google Mantis](https://github.com/google/mantis)**
— and does the one thing that matters before adopting any of them: maps its
stages onto yours, then **scores it with your own held-out key.**

Two things are worth understanding about Mantis specifically.

**It is model-agnostic.** Mantis ships security-review *skills* for coding
agents rather than a bundled model. That is the same architecture as this track:
the pipeline is the product, the model is a component. It means you can run it
on open weights — GLM-4.6, Kimi K2 — which is what makes it usable without a
frontier account.

**It has two output shapes**, and they serve different stages:

- a **`learning_entry`** — appended to a historical learnings file, feeding
  stage 1 (historical parsing) on the next run;
- a **`finding`** object — a vulnerability report, feeding stages 8–10.

That first shape is the interesting one. It closes the loop from Phase 5 back to
Phase 1, which is the property that turns a pipeline into something that
improves.

The bonus framing is deliberate: a reference implementation is a **starting
point you evaluate**, not a product you trust. C2.6 gave you the tools;
this is where you point them at someone else's pipeline.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Map Mantis onto the fifteen stages

Adoption starts with the coverage question: which stages does it do, which does it assume you already have, and which are still yours?

## 4 · Parse the two output shapes

Before scoring anything you have to ingest it. Both shapes are JSON; the `history` field on a learning entry is required and is the one most commonly missing in a first integration.

## 5 · Score it against a held-out key

This is the whole point of the bonus. Conformance is structural — with structured output it goes to 1.00 and says nothing about quality. The number that decides adoption is expert accuracy against a key the tool never saw, matched on **parent directory plus filename**.

## 6 · The stage, as a skill

Google's Mantis is a set of claims: these stages, this output shape, this accuracy. The skill checks all three — maps it onto the stage model, conformance-checks its published samples, and scores its findings against a held-out key.

In [ ]:
# skills/appsec/reference-pipeline-scoring/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: reference-pipeline-scoring
description: >-
  Map a published or vendor pipeline onto the stage model, check its output
  against the schema it claims, and score its findings against a held-out key.
  Use when evaluating a reference implementation, a security product's agent, or
  any pipeline you are asked to adopt.
allowed-tools: Read, Grep, Glob
---

# A reference implementation is something you evaluate

Somebody else's pipeline is a set of claims: these stages, this output shape,
this accuracy. All three are checkable, and checking them is cheaper than
adopting and discovering. The interesting result is usually not the score — it
is the phase the pipeline does not cover at all.

## When to use this

Before adopting a published pipeline, when comparing two, and when a vendor
claims a number you are expected to plan around.

## Procedure

**1 — Map it onto the stage model.** For each stage, does the pipeline cover it
strongly, weakly, or not at all? Coverage claims are usually accurate about the
stages the pipeline is proud of and silent about a whole phase.

**2 — Take its output and check conformance to its own schema.** Required
fields present, enumerated values in range, nothing that is prose where an
object was promised. Conformance failures in a published sample are the cheapest
finding available.

**3 — Score against a held-out key.** Not the examples the pipeline ships. Match
findings to truth by location and defect class, then compute precision and
recall, and report both — a pipeline can look excellent on either alone.

**4 — Separate the model's errors from the harness's.** A null CWE is a schema
problem; a finding at the wrong location is an analysis problem. They have
different fixes and different owners.

**5 — Report the uncovered phase as the headline.** A pipeline that scores well
on the stages it implements is still a partial answer, and the gap is what you
would have to build.

## Output contract

```json
{
  "stages": [{"stage": 0, "coverage": "strong|weak|none"}],
  "uncovered_phases": ["str"],
  "conformance": {"samples": 0, "conforming": 0, "failures": [{"sample": 0, "reason": "str"}]},
  "score": {"matched": 0, "precision": 0.0, "recall": 0.0, "key": "held-out"},
  "errors": {"schema": 0, "analysis": 0}
}
```

## Failure modes

- **Scoring on the shipped examples.** They were chosen.
- **Reporting one of precision and recall.** Either alone flatters.
- **Treating uncovered stages as out of scope.** They are the work you inherit.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/appsec/reference-pipeline-scoring/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/appsec/reference-pipeline-scoring/scripts/reference_pipeline_scoring.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Map a published pipeline onto the stage model and score its output against a held-out key.

This is the executable half of the `reference-pipeline-scoring` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

STAGES = {
 1:  "historical parsing",        2:  "structural indexing",
 3:  "component summarisation",   4:  "architecture synthesis",
 5:  "threat modelling",          6:  "strategic planning",
 7:  "vulnerability auditing",    8:  "deduplication",
 9:  "contextual verification",  10:  "feasibility filtering",
 11: "sandbox replication",      12:  "dynamic exploitation",
 13: "exploit chaining",         14:  "remediation engineering",
 15: "severity calibration and reporting",
}
# Coverage as observed from the project's own documented outputs and skills.
MANTIS = {
 1:  ("yes",     "historical_learnings.jsonl is read on subsequent runs"),
 2:  ("partial", "operates over the agent's code-reading tools"),
 3:  ("partial", "context assembled per review target"),
 4:  ("no",      "assumes you supply the architecture context"),
 5:  ("partial", "review skills encode threat patterns rather than deriving them"),
 6:  ("no",      "you decide what to point it at"),
 7:  ("yes",     "the core: security-review skills emitting finding objects"),
 8:  ("partial", "findings are structured, so dedup is possible downstream"),
 9:  ("partial", "structured output aids verification; you still run the checks"),
 10: ("no",      "reachability is yours"),
 11: ("no",      "no sandbox — it is a review harness, not a DAST"),
 12: ("no",      "static review only"),
 13: ("no",      "no chaining"),
 14: ("partial", "can propose fixes; validation is yours (B2.9)"),
 15: ("partial", "emits severity; calibration against confirmation is yours"),
}
print(f"{'stage':>3}  {'name':34s}{'mantis':10s}note")
print("-" * 96)
for n, name in STAGES.items():
    cov, note = MANTIS[n]
    print(f"{n:>3}  {name:34s}{cov:10s}{note}")
from collections import Counter
c = Counter(v[0] for v in MANTIS.values())
print(f"\ncoverage: {dict(c)}")
print(f"→ Mantis is a strong Phase 3 stage-7 implementation with a stage-1 loop.")
print(f"  Phases 4 and 5 remain yours, which is exactly what B2.6-B2.10 built.")

import json

LEARNING_REQUIRED = ("title", "description", "history")
FINDING_REQUIRED  = ("title", "description", "severity", "file", "cwe")

SAMPLE = [
 # learning_entry — feeds stage 1 on the next run
 '{"type":"learning_entry","title":"owner filter built by concatenation",'
 '"description":"reports queries interpolate the owner parameter",'
 '"history":"introduced in c3d4e5f, fixed once in 2025 and reintroduced"}',
 # finding — feeds stages 8-10
 '{"type":"finding","title":"SQL injection in list_reports","severity":"high",'
 '"file":"src/data/reports.py","cwe":"CWE-89",'
 '"description":"owner is concatenated into the query string"}',
 # a learning entry missing the required history field
 '{"type":"learning_entry","title":"path join in docs",'
 '"description":"docs fetch joins user input"}',
 # a finding with a null field
 '{"type":"finding","title":"traversal","severity":"medium",'
 '"file":"src/data/docs.py","cwe":null,'
 '"description":"name is joined onto the base path"}',
 # not JSON at all
 'I found a SQL injection in the reports module.',
]

def ingest(raw):
    try:
        d = json.loads(raw)
    except json.JSONDecodeError as e:
        return None, f"non-conforming: not JSON ({e.msg})"
    kind = d.get("type")
    required = (LEARNING_REQUIRED if kind == "learning_entry"
                else FINDING_REQUIRED if kind == "finding" else None)
    if required is None:
        return None, f"non-conforming: unknown type {kind!r}"
    missing = [k for k in required if not d.get(k)]
    if missing:
        return None, f"non-conforming: {kind} missing {missing}"
    return d, "conforming"

conforming = []
for raw in SAMPLE:
    obj, note = ingest(raw)
    if obj: conforming.append(obj)
    print(f"{note:58s}{raw[:44]}…")
print(f"\nconformance: {len(conforming)}/{len(SAMPLE)} = {len(conforming)/len(SAMPLE):.2f}")

def path_key(p):
    parts = [x for x in (p or "").replace("\\","/").split("/") if x not in ("",".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")

HELD_OUT = {
 "src/data/reports.py": "CWE-89",
 "src/data/docs.py":    "CWE-22",
 "src/web/handlers.py": "CWE-306",     # a finding Mantis did not report
}

def score(findings, truth):
    expert, rows = 0.0, []
    reported = {}
    for f in findings:
        if f.get("type") != "finding": continue
        reported[path_key(f["file"])] = f
    for path, cwe in truth.items():
        f = reported.get(path_key(path))
        if f is None:
            rows.append((path, "MISSED", 0.0)); continue
        if (f.get("cwe") or "").upper() == cwe:
            rows.append((path, "correct", 1.0)); expert += 1.0
        else:
            rows.append((path, f"right file, wrong class ({f.get('cwe')})", 0.5))
            expert += 0.5
    return {"expert_accuracy": round(expert/len(truth), 4), "rows": rows}

# include the null-cwe finding, re-ingested leniently, to show the half-credit case
lenient = [json.loads(r) for r in SAMPLE if r.startswith("{")]
s = score(lenient, HELD_OUT)
print(f"{'file':26s}{'result':38s}score")
print("-" * 74)
for path, result, pts in s["rows"]:
    print(f"{path:26s}{result:38s}{pts}")
print(f"\nconformance       {len(conforming)/len(SAMPLE):.4f}   ← structural. NOT quality.")
print(f"expert accuracy   {s['expert_accuracy']:.4f}   ← the adoption number")
assert s["expert_accuracy"] < 1.0

# Close the loop: a learning entry feeds stage 1 of the NEXT run.
learnings = [d for d in conforming if d["type"] == "learning_entry"]
print(f"{len(learnings)} learning entry/entries carried into the next run:")
for l in learnings:
    print(f"   {l['title']}")
    print(f"      history: {l['history']}")

def next_run_risk_zones(learnings, findings):
    zones = {}
    for f in findings:
        if f.get("type") == "finding":
            zones[f["file"]] = zones.get(f["file"], 0) + 1
    for l in learnings:
        if "reintroduced" in l.get("history", ""):
            for f in findings:
                if f.get("type") == "finding" and l["title"].split()[0] in f["description"]:
                    zones[f["file"]] = zones.get(f["file"], 0) + 2
    return sorted(zones.items(), key=lambda kv: -kv[1])

print("\nstage 1 input for the next run (Phase 5 → Phase 1):")
for path, weight in next_run_risk_zones(learnings, lenient):
    print(f"   {path:26s} weight {weight}")

print("\nADOPTION CHECKLIST")
for item in [
  "map its stages onto your fifteen — know what it does NOT do",
  "score it against YOUR held-out key before trusting a single finding",
  "report conformance and accuracy separately, always",
  "keep Phase 4 — a static reviewer cannot confirm exploitability",
  "feed learning entries back into stage 1, or the loop does not close",
]:
    print(f"   · {item}")

## What you just proved

The stage map shows Mantis covering stage 7 strongly with a stage-1 learning loop, and not covering Phase 4 at all. Three of five sample outputs conform — one learning entry is missing the required `history` field, one finding has a null CWE, and one is prose. Scored against the held-out key, expert accuracy is below 1.0: one correct, one half credit for the null class, and one missed finding Mantis never reported. The learning entry then feeds the next run's risk zones.

## Your turn

Run the real thing: clone `google/mantis`, point it at a repository you have ground truth for, and score its output with a scoring harness. The gap between its conformance and its expert accuracy on *your* code is the only number that should decide whether you adopt it.

## Where this leaves you

**What you can do now.** A harness you can name the eight parts of, evaluate on a corpus with known answers rather than on how confident it sounds, price per confirmed finding across a run nobody watched, and salt with bait that has no false positives.

**What you still cannot do.** Everything you have built so far is defensive and cooperative: it runs against systems that are not trying to defeat it. You have no evidence about how any of it behaves against someone who is — including the evaluation you have been trusting.

**Function C attacks it, starting with the loop pointed the other way round. Next → C1.0, what red teaming and research with AI means.**

---

**Next → [C1.0 · Start here — what red teaming and research with AI means](https://spbreed.github.io/cyber-commons/lessons/C1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.14.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.14.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*